# 03 Silver: chunking for search

| | |
|---|---|
| **Layer** | Silver |
| **Reads** | `silver_doc_pages` |
| **Writes** | `silver_doc_chunks` |
| **Re-runnable (idempotent)** | Yes. The table is rebuilt deterministically from the pages table. |

Search works on chunks, not whole documents. Each page is split into overlapping windows of words. Chunks never cross page boundaries, so every chunk traces to one page. The overlap keeps a sentence that straddles a boundary findable from either chunk.

**Tuning:** about 200 words is a reasonable default. Smaller chunks are more precise but lose context. Larger chunks keep context but blur the match. Test with real questions, and check the limit of your embedding model.

**Standards followed:** parameters at the top, no secrets in code, small documented functions, explicit schemas, a validation cell at the end, and a printed run summary.

## Parameters

In [ ]:
PAGES_TABLE   = "silver_doc_pages"
CHUNKS_TABLE  = "silver_doc_chunks"
CHUNK_WORDS   = 200    # target chunk size in words
OVERLAP_WORDS = 30     # words shared between neighboring chunks (about 10 to 20 percent)
MIN_WORDS     = 5      # drop fragments shorter than this

## Helpers

In [ ]:
from pyspark.sql import functions as F

CHUNKS_SCHEMA = ("chunk_id string, doc_id string, file_name string, page_number int, chunk_index int, "
                 "chunk_text string, word_count int, batch_id string")


def chunk_words(text: str, size: int, overlap: int) -> list:
    """Split text into windows of `size` words that overlap by `overlap` words."""
    if overlap >= size:
        raise ValueError("overlap must be smaller than size")
    words = text.split()
    chunks, step = [], size - overlap
    for start in range(0, len(words), step):
        chunks.append(" ".join(words[start:start + size]))
        if start + size >= len(words):          # last window reached the end: no tiny trailing chunk
            break
    return chunks


def make_chunk_id(doc_id: str, page: int, index: int) -> str:
    """Readable, stable id: first 12 hash characters, page, chunk number."""
    return f"{doc_id[:12]}-p{page:03d}-c{index:02d}"

## Build chunks

In [ ]:
rows = []
for page in spark.table(PAGES_TABLE).select("doc_id", "file_name", "page_number", "text", "batch_id").collect():
    for index, piece in enumerate(chunk_words(page["text"] or "", CHUNK_WORDS, OVERLAP_WORDS)):
        if len(piece.split()) >= MIN_WORDS:
            rows.append((make_chunk_id(page["doc_id"], page["page_number"], index), page["doc_id"],
                         page["file_name"], page["page_number"], index, piece, len(piece.split()), page["batch_id"]))

chunks = spark.createDataFrame(rows, CHUNKS_SCHEMA)
chunks.write.format("delta").mode("overwrite").saveAsTable(CHUNKS_TABLE)   # derived table: safe to rebuild
print("chunks written:", chunks.count())

## Validation

In [ ]:
chunks = spark.table(CHUNKS_TABLE)
assert chunks.select("chunk_id").distinct().count() == chunks.count(), "chunk_id must be unique"
display(chunks.agg(F.min("word_count"), F.avg("word_count"), F.max("word_count")))
display(chunks.groupBy("file_name").count())